# musubi-tuner — Convert LoRA Format

Converts `.safetensors` LoRA files between **musubi-tuner** format and **ComfyUI / Diffusers** format.

| Direction | `--target` | Use case |
|---|---|---|
| musubi-tuner → ComfyUI/Diffusers | `other` | Use your trained LoRA in ComfyUI |
| ComfyUI/Diffusers → musubi-tuner | `default` | Import an external LoRA for further training |

Supports LoRA, LoHa, and LoKr formats.

---

**Workflow:**
1. Upload your `.safetensors` file(s) to Google Drive → `Convert/Input/`
2. Run all cells (no GPU needed — use CPU runtime to save quota)
3. Collect converted files from `Convert/Output/`


## Section 1 — Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted.")


## Section 2 — Install musubi-tuner

Only needs the Python package — no GPU required for conversion.


In [ ]:
import os

REPO_DIR = "/content/musubi-tuner"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kohya-ss/musubi-tuner {REPO_DIR} -q
else:
    !git -C {REPO_DIR} pull -q

%cd {REPO_DIR}
!pip install -q -e . --no-deps
!pip install -q safetensors torch

print("✅ Ready.")


## Section 3 — Configuration

**Edit `CONVERSION_DIRECTION`** to match what you need.


In [ ]:
# ============================================================
#   PATHS — adjust if your Drive layout differs
# ============================================================
DRIVE_ROOT   = "/content/drive/MyDrive/musubi-tuner-collab"
INPUT_DIR    = f"{DRIVE_ROOT}/Convert/Input"
OUTPUT_DIR   = f"{DRIVE_ROOT}/Convert/Output"

# ============================================================
#   CONVERSION DIRECTION
#     "other"   → musubi-tuner format  ➜  ComfyUI / Diffusers
#     "default" → ComfyUI / Diffusers  ➜  musubi-tuner format
# ============================================================
CONVERSION_DIRECTION = "other"

# ============================================================
#   OPTIONAL — leave None unless you know you need it
#   Sets the Diffusers weight prefix (default: "diffusion_model")
# ============================================================
DIFFUSERS_PREFIX = None

# ============================================================
#   OVERWRITE existing output files?
# ============================================================
OVERWRITE = False

import os
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

direction_label = {
    "other":   "musubi-tuner → ComfyUI / Diffusers",
    "default": "ComfyUI / Diffusers → musubi-tuner",
}
print(f"✅ Config loaded.")
print(f"   Direction : {direction_label[CONVERSION_DIRECTION]}")
print(f"   Input     : {INPUT_DIR}")
print(f"   Output    : {OUTPUT_DIR}")
print(f"   Overwrite : {OVERWRITE}")


## Section 3.5 — Inspect LoRA Format *(optional but recommended)*

Run this before converting to understand the key format of your LoRA file.

| Detected prefix | Format | Typical source |
|---|---|---|
| `lora_unet_` | musubi-tuner | musubi-tuner training |
| `diffusion_model.` | Diffusers/ComfyUI | ComfyUI, older Diffusers |
| `transformer.` | Diffusers/PEFT | AI Toolkit, newer Diffusers |
| `lora_te` | Kohya/A1111 | sd-scripts, AUTOMATIC1111 |

Also shows whether **alpha keys** are present — missing alphas cause incorrect LoRA scaling (output looks like static or noise instead of a recognizable image).

In [ ]:
import os, glob
from collections import Counter
from safetensors import safe_open

# ── pick file(s) to inspect ────────────────────────────────────────────
files_to_inspect = sorted(glob.glob(os.path.join(INPUT_DIR, '*.safetensors')))

if not files_to_inspect:
    print(f'No .safetensors files found in {INPUT_DIR}')
else:
    for path in files_to_inspect:
        print(f'\n{"="*60}')
        print(f'File : {os.path.basename(path)}')
        print(f'Size : {os.path.getsize(path)/1024**2:.1f} MB')
        print(f'{"="*60}')

        with safe_open(path, framework='pt') as f:
            meta = f.metadata() or {}
            keys = list(f.keys())

        if meta:
            print('\n--- Metadata ---')
            for k, v in meta.items():
                print(f'  {k}: {v}')

        keys.sort()
        prefixes = Counter(k.split('.')[0] if '.' in k else k.split('_')[0]+'_' for k in keys)

        # detect format
        if any(k.startswith('lora_unet_') for k in keys):
            fmt = 'musubi-tuner  →  use --target other to convert to ComfyUI/Diffusers'
        elif any(k.startswith('diffusion_model.') for k in keys):
            fmt = 'Diffusers/ComfyUI (diffusion_model prefix)  →  already usable in ComfyUI'
        elif any(k.startswith('transformer.') for k in keys):
            fmt = 'Diffusers/PEFT (transformer prefix)  →  AI Toolkit style'
        elif any(k.startswith('lora_te') for k in keys):
            fmt = 'Kohya/A1111 format'
        else:
            fmt = 'Unknown / non-standard'

        alpha_keys = [k for k in keys if k.endswith('.alpha')]

        print(f'\n--- Analysis ---')
        print(f'  Total keys : {len(keys)}')
        print(f'  Format     : {fmt}')
        print(f'  Prefixes   : {dict(prefixes)}')
        print(f'  Alpha keys : {len(alpha_keys)} ({"present ✅" if alpha_keys else "MISSING ⚠️  — likely cause of static/noise output"})')

        print(f'\n--- First 10 keys ---')
        for k in keys[:10]:
            print(f'  {k}')
        if len(keys) > 10:
            print(f'  ... ({len(keys)-10} more)')


## Section 4 — Scan Input Files

Lists all `.safetensors` files found in `INPUT_DIR`. Review before converting.


In [ ]:
import os, glob

input_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.safetensors")))

if not input_files:
    print(f"⚠️  No .safetensors files found in:")
    print(f"   {INPUT_DIR}")
    print()
    print("Upload your LoRA file(s) there and re-run this cell.")
else:
    print(f"Found {len(input_files)} file(s) to convert:")
    for f in input_files:
        size_mb = os.path.getsize(f) / 1024**2
        out_path = os.path.join(OUTPUT_DIR, os.path.basename(f))
        status = "[exists]" if os.path.exists(out_path) and not OVERWRITE else ""
        print(f"  {os.path.basename(f):50s}  {size_mb:7.1f} MB  {status}")


## Section 5 — Convert

Converts all files found above. Skips files whose output already exists (unless `OVERWRITE = True`).


In [ ]:
import os, glob, sys

REPO_DIR = "/content/musubi-tuner"
os.chdir(REPO_DIR)

input_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.safetensors")))

if not input_files:
    print("No input files — run Section 4 first.")
    sys.exit(0)

converted, skipped, failed = [], [], []

for in_path in input_files:
    fname    = os.path.basename(in_path)
    out_path = os.path.join(OUTPUT_DIR, fname)

    if os.path.exists(out_path) and not OVERWRITE:
        print(f"  ⏭️  {fname} — output exists, skipping")
        skipped.append(fname)
        continue

    print(f"  ⚙️  Converting: {fname}")

    prefix_flag = f"--diffusers_prefix {DIFFUSERS_PREFIX}" if DIFFUSERS_PREFIX else ""
    cmd = (
        f"python src/musubi_tuner/convert_lora.py"
        f" --input \"{in_path}\""
        f" --output \"{out_path}\""
        f" --target {CONVERSION_DIRECTION}"
        f" {prefix_flag}"
    )

    ret = os.system(cmd)
    if ret == 0:
        out_mb = os.path.getsize(out_path) / 1024**2
        print(f"     ✅ Saved: {fname}  ({out_mb:.1f} MB)")
        converted.append(fname)
    else:
        print(f"     ❌ Failed: {fname} (exit code {ret})")
        failed.append(fname)

print()
print("=" * 50)
print(f"Converted : {len(converted)}")
print(f"Skipped   : {len(skipped)}")
print(f"Failed    : {len(failed)}")
if converted:
    print(f"\nOutput files in: {OUTPUT_DIR}")
if failed:
    print(f"\n⚠️  Failed files: {failed}")
    print("   Check scroll-up for error messages.")


## Section 6 — Download Results *(optional)*

Files are already in Google Drive (`Convert/Output/`).  
Uncomment the block below to also download them directly to your browser.


In [ ]:
import os, glob

out_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.safetensors")))

if not out_files:
    print("No output files found.")
else:
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"Files:")
    for f in out_files:
        print(f"  {os.path.basename(f)}  ({os.path.getsize(f)/1024**2:.1f} MB)")

# Uncomment to download to browser:
# from google.colab import files
# for f in out_files:
#     files.download(f)
